**Now we will loads the saved DataFrame back  called "df_eda.pkl"** 

In [45]:
import pandas as pd
df_eda = pd.read_pickle("df_eda.pkl")
df_eda

,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,...,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason,Churn
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,2,...,65.60,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN,0
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,0,...,-4.00,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN,0
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,0,...,73.90,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices,1
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.014457,-122.115432,1,...,98.00,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction,1
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.227846,-119.079903,3,...,83.90,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7037,9986-BONCE,Female,36,No,0,Fallbrook,92028,33.362575,-117.299644,0,...,20.95,85.50,0.00,0,8.04,93.54,Churned,Competitor,Competitor made better offer,1
7038,9987-LUTYD,Female,20,No,0,La Mesa,91941,32.759327,-116.997260,0,...,55.15,742.90,0.00,0,606.84,1349.74,Stayed,NaN,NaN,0
7039,9992-RRAMN,Male,40,Yes,0,Riverbank,95367,37.734971,-120.954271,1,...,85.10,1873.70,0.00,0,356.40,2230.10,Churned,Dissatisfaction,Product dissatisfaction,1
7041,9993-LHIEB,Male,21,Yes,0,Solana Beach,92075,33.001813,-117.263628,5,...,67.85,4627.65,0.00,0,142.04,4769.69,Stayed,NaN,NaN,0


# SQL #

$Step-1$: **load the DataFrame into a SQLite database**

In [46]:
# "sqlite3" is a python built in libreary
# 

import sqlite3

conn = sqlite3.connect(":memory:")           # creates a temporary database in RAM and stores the connection in conn
df_eda.to_sql("customers", conn, index=False, if_exists="replace")

6589

In [47]:
pd.read_sql("SELECT COUNT(*) AS total_rows FROM customers", conn) # here Select the customers dataset. 
# count will count how many rows are there, for that table name = total_rows

,total_rows
0,6589


$Step-2$: **churn rate by contract type (GROUP BY)**

In [48]:
query = """
 SELECT
    Contract,
    COUNT(*) AS customers,
    SUM(Churn) AS churned,                                       
    ROUND(100.0 * SUM(Churn) / COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY Contract
ORDER BY churn_rate_pct DESC
"""
pd.read_sql(query, conn)

# query = """       """  stores the SQL in a Python variable
# SELECT Contract, ... chooses the column  you want 
# FROM customers says which table to read.
# GROUP BY Contract : it will make groups like "Month-to-Month", "One Year", "Two Year"
# COUNT(*) : counts the customers in each group
# SUM(Churn) : counts the churned customers in each group.
# ROUND(..., 1) : keeps one decimal place.
# ORDER BY churn_rate_pct DESC : sorts the highest churn rate first

,Contract,customers,churned,churn_rate_pct
0,Month-to-Month,3202,1655,51.7
1,One Year,1526,166,10.9
2,Two Year,1861,48,2.6


$Step-3$: **a subquery (compare churn above and below the average bill)**

- Subquery:  A subquery is a query inside another query

In [49]:
query = """
SELECT
    CASE
        WHEN "Monthly Charge" > (SELECT AVG("Monthly Charge") FROM customers)
        THEN 'Above average'
        ELSE 'Average or below'
    END AS charge_group,
    COUNT(*) AS customers,
    SUM(Churn) AS churned,
    COUNT(*) - SUM(Churn) AS stayed,
    ROUND(100.0 * SUM(Churn)/ COUNT(*), 1) AS churn_rate_pct
FROM customers
GROUP BY charge_group
"""
pd.read_sql(query, conn)

# (SELECT AVG("Monthly Charge") FROM customers) is the subquery It runs first and returns one number
# CASE: means where subquery started. END AS charge_group: Here subquery  end with label charge_group
# GROUP BY charge_group: this will group the  THEN: 'Above average' ELSE: 'Average or below'. 2 gropus
#  COUNT(*) - SUM(Churn) AS stayed: when we do this we will get how many peoples are stayed in service

,charge_group,customers,churned,stayed,churn_rate_pct
0,Above average,3726,1337,2389,35.9
1,Average or below,2863,532,2331,18.6


- Above average bill: 3726 customers, and 1337 of them churned. 1337 ÷ 3726 = 35.9%. So roughly 36 out of every 100 customers in this group left.

- Average or below: 2863 customers, and 532 of them churned. 532 ÷ 2863 = 18.6%. So roughly 19 out of every 100 customers in this group left.

- In EDA we compared the average monthly charge of the two kinds of customers:

    - customers who churned paid 73.35 on average.
    
    - customers who stayed paid 61.74 on average

$Step-4$: **a JOIN**

In [50]:
# A JOIN needs two tables, and so far we only have customers

contract_risk = pd.DataFrame({ "Contract" : ["Month-to-Month","One Year","Two Year"],
                               "risk_level": ["High","Medium","Low"]
                             })
contract_risk.to_sql("contract_risk", conn, index=False, if_exists="replace")

# second table name = contract_risk

3

In [51]:
query = """
SELECT
    r.risk_level,
    COUNT(*) AS customers,
    SUM(c.Churn) AS churned,
    COUNT(*) - SUM(c.Churn) AS Stayed,
    ROUND(100.0 * SUM(c.Churn)/ COUNT(*), 1) AS churn_rate_pct
FROM customers c
JOIN contract_risk r
    ON c.Contract = r.Contract
GROUP BY r.risk_level
ORDER BY churn_rate_pct DESC
"""

pd.read_sql(query, conn)

# here named customer table as "c"                 named contract_risk table as "r" . Both  tables are joined by using "JOIN"
# ON c.Contract = r.Contract is the matching rule. SQL finds the row in contract_risk with the same "contract type" and attaches its risk_level.
# Contract types (Month-to-Month, One Year, Two Year)
# How it will attaches to risk_level ? Ans: Month-to-Month = High, One Year = Medium, Two Year = Low 
# then group based on "risk_level" (groups= High, Medium, Low)
# then count, sum will perform

,risk_level,customers,churned,Stayed,churn_rate_pct
0,High,3202,1655,1547,51.7
1,Medium,1526,166,1360,10.9
2,Low,1861,48,1813,2.6


- GROUP BY: churn rate by contract type
- Subquery: above-average vs average-or-below bill (35.9% vs 18.6%)
- JOIN: churn rate by contract risk level

- Month-to-Month contracts churn about 20 times more than Two Year contracts (51.7% vs 2.6%)

- Customers with above-average bills churn almost twice as often (35.9% vs 18.6%)

$Step-5$: **top 5 reasons customers churned**

In [52]:
query = """
SELECT 
    "Churn Category",
    "Churn Reason",
    COUNT(*) AS churned_customers
FROM customers
WHERE Churn = 1
GROUP BY "Churn Category", "Churn Reason"
ORDER BY  churned_customers DESC
LIMIT 5
"""

pd.read_sql(query, conn)

,Churn Category,Churn Reason,churned_customers
0,Competitor,Competitor had better devices,313
1,Competitor,Competitor made better offer,311
2,Attitude,Attitude of support person,220
3,Other,Don't know,130
4,Competitor,Competitor offered more data,117


- Top churn reasons are competitor-related (better devices, better offer, more data)

- Three of the top five reasons are competitor-related: better devices (313), a better offer (311) and more data (117). Together that is 741 churned customers.
  
- The only reason in the top five that your own company controls directly is “Attitude of support person” (220), which is a service-quality problem.

- “Don’t know” (130) is a reason with no information in it.